## 1. 지능형 용접 공정: 3×3 디지털 트윈 대시보드 환경 구축

본 절은 로봇 아크 용접 공정을 컴퓨터 환경 내에서 재현하는 디지털 트윈(Digital Twin) 기반 시뮬레이션 시스템의 구축 과정을 기술한다. 용접 중 발생하는 복합적인 물리적 변동성(Multi-physics Disturbance)을 3×3 형태의 9분할 다중 패널로 실시간 모니터링하고, 인공지능(AI) 제어기가 자율적으로 결함을 억제하는 폐루프(Closed-loop) 제어 아키텍처의 타당성을 검증하는 것을 목적으로 한다.

본 셀에서는 다물리(Multi-physics) 시뮬레이션 및 데이터 시각화를 위한 기초 컴퓨팅 환경을 설정한다.

* **종속성 패키지(Dependencies):** 고속 배열 연산을 위한 `numpy`, 시계열 데이터 처리를 위한 `pandas`, 실시간 동적 렌더링 및 대시보드 시각화를 위한 `matplotlib` 라이브러리를 로드한다.
* **재현성(Reproducibility):** 난수 시드(`SEED=2026`)를 고정하여 확률적 외란 생성 과정의 재현 가능성을 확보한다.
* **유의사항:** 본 시뮬레이션이 다루는 모든 공정 변수(전류, 전압, 표면온도, 접촉각 등)의 시계열은 실측 센서 데이터가 아니라 `numpy.random`을 이용해 확률적으로 생성한 합성(Synthetic) 데이터이다. 즉 본 노트북은 실측 용접 데이터에 대한 분석이 아니라, 폐루프 제어 알고리즘의 개념 증명(Proof-of-Concept)을 위한 수치 시뮬레이션이다.

In [ ]:
# ==========================================
# [Cell 1] 3x3 대시보드를 위한 라이브러리 및 환경 설정
# ==========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Ellipse
import matplotlib.patches as patches
from IPython.display import HTML

# Windows 환경 한글 폰트 및 마이너스 부호 깨짐 방지
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-darkgrid')

# 결과 저장 디렉토리 생성
RESULT_DIR = "result_sim"
os.makedirs(RESULT_DIR, exist_ok=True)

# 재현성을 위한 시드 고정
SEED = 2026
np.random.seed(SEED)

print("✅ [초기화 완료] 3x3 대시보드 시뮬레이션 환경 설정 완료")

## 2. 조인트 형상 역학 및 다중 외란(Disturbance) 모델링

후판(3 mm 이상) 용접 시 완전 용입(Full Penetration)을 확보하기 위해서는 모재(Base Metal) 단부에 V-그루브(V-Groove, 개선 가공)를 적용해야 한다. 이를 통해 용융풀(Weld Pool)이 루트(Root) 하단까지 원활히 침투할 수 있는 기하학적 조건이 형성된다.

---

```text
  [용접 토치 / 와이어] 
       │
       ▼ (용융 금속 이행)

    ＼           ／  <-- 개선각 (Bevel Angle)

────＼           ／────
 모재 1 │ ── 틈 ── │ 모재 2
     │          │
     └────┬─────┘
          ▲
   루트 갭 (Root Gap)
```

---

### 주요 용접 품질 결정 기하학적 인자

1. **루트 갭(Root Gap):** 모재 하단부의 이격 거리를 의미한다.
   * *과소(< 1 mm):* 용융 금속의 침투 경로가 제한되어 **용입 부족(Lack of Penetration)** 결함을 유발한다.
   * *과다(> 3 mm):* 용융풀의 지지력이 상실되어 **용락(Burn-through)** 결함을 유발한다.
2. **개선각(Bevel Angle):** 그루브 경사면의 각도를 의미한다.
   * *과소(< 20°):* 아크 도달 범위가 제한되어 **융합 불량(Lack of Fusion)**을 유발한다.
   * *과다(> 45°):* 과도한 용착 금속량이 요구되고 열이 누적되어 **열변형(Thermal Distortion)**을 유발한다.

### 전자기 아크 필드(Electromagnetic Arc Field) 방정식

토치 끝단과 모재 간 아크 길이($l_a$)의 변동은 아크 전압($V$)과 직결된다.

$$V=V_0+E\cdot l_a\quad[\text{V}]$$

* $l_a$: 토치 진동 등으로 발생하는 아크 길이의 물리적 편차 $[\text{mm}]$.
* $V_0$: 통전에 필요한 최소 전압 강하(기본 전압) $[\text{V}]$.
* $E$: 아크 기둥의 전기장 세기(전계 강도) $[\text{V/mm}]$.

통계역학(Statistical Mechanics)에서 다체계(Many-body System)의 열요동을 취급하는 것과 마찬가지로, 본 시뮬레이션에서는 공정 중 발생하는 갭 변동, 판재 굴곡, 토치 진동 등을 시스템의 불안정성을 야기하는 **확률론적 다차원 외란(Stochastic Disturbance)**으로 정의한다. 본 셀에서는 이러한 비선형적 노이즈를 30초 구간에 걸쳐 생성하여 동적 상태 공간(State Space)을 구축한다. 후술하는 바와 같이, 이 외란 중 표면온도 편차 및 접촉각(굴곡) 편차는 **평균 0의 가우시안 증분이 누적되는 랜덤워크(Random Walk, `cumsum`) 형태로 생성되어 시간에 따라 분산이 무한히 증가하는 비정상(Non-stationary) 확률과정**이라는 점에 유의해야 한다 — 이는 5절에서 다루는 결함 확률 거동의 근본 원인이 된다.

In [ ]:
# ==========================================
# [Cell 2] 다물리 상태 공간 엔진 (Multi-Physics State Engine)
# ==========================================
class MultiPhysicsStateEngine:
    def __init__(self, steps=300, dt=0.1): # 300스텝 (30초) 진행
        self.steps = steps
        self.dt = dt
        self.time = np.arange(steps) * dt
        
        # 기계 및 기하학적 기준값 (표준값)
        self.v0 = 8.0              # 기준 용접 속도 (mm/s)
        self.theta0 = 15.0         # 기준 토치 각도 (°)
        self.dist0 = 12.0          # 기준 팁-모재 간 거리 (mm)
        self.Tp0 = 2.0             # 기준 판재 두께 (mm)
        self.P0 = 60.0             # 기준 가압력 (psi)
        self.T0 = 25.0             # 기준 표면 온도 (°C)
        
        # 전기 및 열역학적 기준값 (표준값)
        self.I0 = 200.0            # 기준 전류 (A)
        self.V0 = 24.0             # 기준 전압 (V)
        self.eta = 0.8             # 열효율
        self.alpha_R = 0.004       # 저항 온도계수
        self.target_Q = (self.eta * self.V0 * self.I0) / self.v0 # 기준 열입력 (480 J/mm)
        
    def generate_telemetry(self):
        """다차원 센서 외란(Disturbance) 시뮬레이션 데이터 생성"""
        n = self.steps
        # 확률적 노이즈 누적 (환경의 변화)
        self.curv_dev = np.cumsum(np.random.normal(0, 0.4, n)) + np.random.normal(0, 1.5, n)
        self.temp_dev = np.cumsum(np.random.normal(0, 1.5, n)) + np.random.normal(0, 3, n)
        
        # 센서 실시간 관측 배열
        self.theta_eff = self.theta0 + self.curv_dev
        self.dist_eff = self.dist0 + np.random.normal(0, 1.0, n)
        self.T_surf = self.T0 + self.temp_dev
        self.P_eff = self.P0 + np.random.normal(0, 2.0, n)
        
        # I-V 공간 노이즈 (전원 장치의 불안정성)
        self.I_obs = self.I0 + np.cumsum(np.random.normal(0, 0.8, n)) + np.random.normal(0, 2.0, n)
        self.V_obs = self.V0 + np.cumsum(np.random.normal(0, 0.1, n)) + np.random.normal(0, 0.5, n)
        self.Tp_obs = np.full(n, self.Tp0) + np.random.normal(0, 0.05, n)

engine = MultiPhysicsStateEngine(steps=300, dt=0.1)
engine.generate_telemetry()
print(f"✅ [데이터 생성] {engine.steps} 스텝의 다차원 환경 및 장비 변동성 모델링 완료")

## 3. 공정 물리 지배 방정식 및 적응형 AI 폐루프(Closed-loop) 제어

용접 공정은 전기 에너지 입력, 열역학적 상변화, 용융 금속의 물리적 이행으로 이어지는 물리 법칙의 지배를 받는다.

### 1) 단위 길이당 유효 열입력(Thermodynamic Heat Input)

$$Q=\eta\cdot\frac{V\cdot I}{v}\quad[\text{J/mm}]$$

* $Q$: 단위 길이당 실질적으로 전달된 유효 열입력.
* $\eta$: 전기 에너지가 열 에너지로 변환·흡수되는 열효율(통상 0.8).
* $V$: 용접 전압(비드 폭 및 아크의 공간적 팽창을 결정하는 인자).
* $I$: 용접 전류(아크 열량 및 용융 깊이를 결정하는 인자).
* $v$: 토치 이동 속도 — 열입력($Q$)과 반비례 관계를 가지는 **핵심 제어 조작 변수(Manipulated Variable, MV)**이다.

### 2) 용착 속도 및 질량 보존 방정식(Mass Balance & Joule Heating)

공급되는 와이어 송급 속도(WFS)와 아크에 의해 용융되는 속도(Melting Rate, MR)는 동적 평형(Dynamic Equilibrium) 상태를 유지해야 한다.

$$WFS=MR=\alpha\cdot I+\beta\cdot L_e\cdot I^2\quad[\text{m/min}]$$

* $\alpha\cdot I$: 아크 복사열에 의한 용융 성분(전류 $I$에 1차 비례).
* $\beta\cdot L_e\cdot I^2$: 와이어 내부 저항을 통과하는 전류의 줄 발열(Joule Heating) 성분(전류 $I^2$에 2차 비례).
* **동적 평형:** 전류($I$) 상승 시 와이어 용융 속도가 비선형적으로 급증하므로, 파단을 방지하기 위해서는 와이어 송급 속도($WFS$)의 실시간 동기화가 필수적이다.

### 3) 실시간 AI 기반 능동형 가드레일(Active Closed-loop Guardrail)

1. **[상태 관측(Sensing)]** 모재의 표면온도 변동 및 조인트 이격 거리(접촉각)의 기하학적 오차를 0.1초 간격으로 실시간 관측한다.
2. **[상태 추정(Estimation)]** 해당 외란 조건에서 초과 입열에 의한 용락(Burn-through) 또는 과소 입열에 의한 결함 발생 확률을 로지스틱 함수로 추정한다.
3. **[보상 제어(Compensation)]** 열입력 지배 방정식($Q=\eta VI/v$)에 근거하여, 유효 열입력이 허용 범위($Q_{min}=0.85Q_0 \sim Q_{max}=1.15Q_0$)를 벗어나거나 유효 접촉각이 임계치(25°)를 초과하는 경우, 제어기는 조작 변수인 토치 이동 속도($v$)를 즉각 재산정하여 유효 열입력을 목표값($Q_0$)으로 강제 복귀시킨다.

**설계상의 범위(Scope) 한정:** 본 폐루프 제어기의 조작 변수는 토치 이동 속도 $v$ 단일 변수로 한정된다. 즉 제어기는 전류·전압의 확률적 요동이나 접촉각(굴곡) 자체의 물리적 편차를 직접 교정하는 구동기(Actuator)를 보유하지 않으며, 오직 그 결과로 나타나는 **유효 열입력 $Q_{eff}$ 한 가지 지표만을 목표값으로 되돌리는 단일변수(Single-variable) 보상 체계**이다. 이 구조적 한정은 5절의 결과 분석에서 관측되는 지표 간 상반된 거동(열입력의 안정적 수렴 vs. 결함 확률의 지속적 진동)을 해석하는 핵심 단서를 제공한다.

In [ ]:
# ==========================================
# [Cell 3] 폐루프 제어 및 용접 형상/품질 연산기
# ==========================================
def adaptive_closed_loop_control(eng):
    n = eng.steps
    res = {
        'x_pos': np.zeros(n), 'v_corr': np.full(n, eng.v0), 'Q_eff': np.zeros(n),
        'W': np.zeros(n), 'D': np.zeros(n), 'd_nugget': np.zeros(n),
        'F_pull': np.zeros(n), 'P_defect': np.zeros(n), 'is_active': np.zeros(n, dtype=bool)
    }
    
    Q_min, Q_max = eng.target_Q * 0.85, eng.target_Q * 1.15
    theta_max = 25.0 
    
    for i in range(n):
        # 1. 유효 저항 및 초기 열입력 연산
        R_factor = (1.0 + eng.alpha_R * (eng.T_surf[i] - eng.T0)) * (eng.P0 / eng.P_eff[i])
        current_Q = (eng.eta * eng.V_obs[i] * eng.I_obs[i] * R_factor) / eng.v0
        
        # 2. 물리 가드레일 (속도 v 보정)
        if current_Q > Q_max or current_Q < Q_min or eng.theta_eff[i] > theta_max:
            res['v_corr'][i] = (eng.eta * eng.V_obs[i] * eng.I_obs[i] * R_factor) / eng.target_Q
            res['Q_eff'][i] = eng.target_Q
            res['is_active'][i] = True
        else:
            res['Q_eff'][i] = current_Q
            res['is_active'][i] = False
            
        # 3. 비드 폭(W)과 깊이(D), 너겟 직경 연산
        res['W'][i] = np.sqrt(res['Q_eff'][i]) * 0.4
        res['D'][i] = (res['Q_eff'][i] ** 0.8) * 0.04
        res['d_nugget'][i] = 0.25 * np.sqrt(res['Q_eff'][i]) * (eng.Tp_obs[i] / eng.Tp0)
        
        # 4. 결함 확률 및 강도 예측
        risk_score = 0.1 * abs(res['Q_eff'][i] - eng.target_Q) + 0.5 * max(0, eng.theta_eff[i] - 15.0)
        res['P_defect'][i] = 1.0 / (1.0 + np.exp(-(risk_score - 5.0)))
        res['F_pull'][i] = (np.pi / 4) * (res['d_nugget'][i]**2) * 400.0 * (1.0 - res['P_defect'][i]) * 0.01

    res['x_pos'] = np.cumsum(res['v_corr'] * eng.dt)
    return res

res = adaptive_closed_loop_control(engine)
target_W = np.sqrt(engine.target_Q) * 0.4
target_D = (engine.target_Q ** 0.8) * 0.04

print("✅ [제어 완료] 비드 형상, 강도, 결함 확률 등 모든 지표 산출 완료")

## 4. 공차 기반 무차원 정규화 및 3×3 디지털 트윈 대시보드 구성

온도(°C), 각도(°), 형상 치수(mm), 전압(V) 등 이질적인 측정 단위를 갖는 다차원 물리량을 통합적으로 모니터링하기 위하여, 각 변수 고유의 안전 허용 공차를 반영한 **무차원 정규화 지수(Normalized Tolerance Index, $I_{norm}$)**를 도입한다.

### 공차 기반 정규화 지수(Tolerance-based Normalization Index)

$$I_{norm} = 100 + 10 \cdot \left( \frac{X - X_0}{\Delta X_{safe}} \right) \quad [\%]$$

* $X$: 실시간 측정 변수.
* $X_0$: 공정 표준(Target) 설정값.
* $\Delta X_{safe}$: 해당 변수가 기계적·물리적 안전성을 유지할 수 있는 고유 허용 공차 한계(Tolerance Margin).

| 상태 분류 | 정규화 지수($I_{norm}$) | 범례 색상 | 공정 상태 해석 |
| :---: | :---: | :---: | :--- |
| **안전(Safe)** | $90\% \le I_{norm} \le 110\%$ | 🟢 초록색 | 변동성이 제어 허용 오차 내에 존재(정상 작동) |
| **경계(Caution)** | $75\%\sim 90\%$ 또는 $110\%\sim 125\%$ | 🟡 노란색 | 임계 마진 도달. 시스템 모니터링 강화 요망 |
| **위험(Risk)** | $I_{norm} < 75\%$ 또는 $I_{norm} > 125\%$ | 🔴 빨간색 | 공차 이탈 구간. 결함 발생 위험 및 AI 제어기의 즉각적 보상 개입 궤적 활성화 |

---

### 3×3 디지털 트윈 대시보드 시각화 매핑 지표

1. **[1행 1열] 선형 토치 궤적 및 냉각 모델:** 토치의 공간적 진동 궤적 추적 및 1,500°C 이상의 용융풀(Weld Pool)이 응고되는 2차원 열전달·냉각 프로파일 렌더링.
2. **[1행 2열] 비드 및 너겟 단면(Cross-section):** 용접부 루트 방향 침투 깊이 및 너겟 단면적의 기하학적 일관성 추적.
3. **[1행 3열] 환경 변수 정규화 상태:** 표면 예열 온도, 토치 지향 각도, 목표 깊이 및 폭에 대한 실시간 상태 지수 및 최대·최소 오차 마진($I_{norm}$) 모니터링.
4. **[2행 1열] V-I 위상 다이어그램(Phase Diagram):** 전압-전류가 구성하는 2차원 위상 공간 상의 확률 밀도 등고선(KDE) 및 동적 동작점 궤적 감시.
5. **[2행 2열] 유효 열입력 $Q_{eff}(t)$:** 공정 품질을 지배하는 핵심 열역학적 척도의 실시간 추이 및 제어 안정성 검증.
6. **[2행 3열] 장비 제어 변수 정규화 상태:** 전원 장치의 상태 변수(V, I)와 기계적 조작 변수(Speed)의 피드백 루프 균형 상태 가시화.
7. **[3행 1열] 조작 변수(MV) 제어 속도 $v(t)$:** 물리적 외란 감지 시 제어기가 토치 이동 속도를 동적으로 가변 보상하는 실시간 응답 궤적.
8. **[3행 2열] 기계적 물성 예측 $F_{pull}(t)$:** 응고 완료된 접합부의 구조적 건전성을 대변하는 인장 전단 강도 실시간 추정치.
9. **[3행 3열] 결함 확률 및 가드레일 방어 성능:** 적응형 제어 작동 결과에 따른 최종 로지스틱 결함 발생 확률의 시계열 궤적.

In [ ]:
# ==========================================
# [Cell 4] 3x3 패널 디지털 트윈 애니메이션 렌더링 
# (하단 슬라이더 제거 및 그래프 여백 최적화 완비)
# ==========================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import scipy.ndimage as ndimage
from IPython.display import HTML

# 1. 주피터 애니메이션 용량 제한 넉넉하게 해제
plt.rcParams['animation.embed_limit'] = 500

# 2. 모든 그래프의 텍스트(제목, 축 라벨, 축 숫자) 크고 굵게 전역 설정
plt.rcParams.update({
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.titlesize': 17,    
    'axes.labelsize': 15,    
    'xtick.labelsize': 14,   
    'ytick.labelsize': 14,   
    'legend.fontsize': 13    
})

fig, axes = plt.subplots(3, 3, figsize=(24, 20), dpi=90) 
fig.suptitle('Ultimate 3x3 Digital Twin: Adaptive Welding Process Simulation', fontsize=24, fontweight='bold')
# 슬라이더가 제거되었으므로 bottom 여백을 0.08에서 0.05로 줄여 공간 최적화
plt.subplots_adjust(bottom=0.05, hspace=0.45, wspace=0.35)

# ------------------------------------------
# [애니메이션 재생 속도 및 축 범위 동적 설정]
# ------------------------------------------
ANIMATION_FPS = 10
FRAME_INTERVAL = int(1000 / ANIMATION_FPS)

MAX_TIME = engine.steps * engine.dt
MAX_X_POS = max(res['x_pos']) * 1.05

target_W = np.sqrt(engine.target_Q) * 0.4
target_D = (engine.target_Q ** 0.8) * 0.04

# 전체 데이터 기준 정규화 인덱스 사전 연산
env_ratios_all = np.zeros((4, engine.steps))
mach_ratios_all = np.zeros((4, engine.steps))
TOL_ENV = [5.0, 3.0, target_D * 0.1, target_W * 0.15] 
TOL_MACH = [1.0, 15.0, 1.0, engine.target_Q * 0.05]   

for j in range(engine.steps):
    env_vals = [engine.T_surf[j], engine.theta_eff[j], res['D'][j], res['W'][j]]
    env_refs = [engine.T0, engine.theta0, target_D, target_W]
    env_ratios_all[:, j] = [100 + 10 * (v - ref) / tol for v, ref, tol in zip(env_vals, env_refs, TOL_ENV)]
    
    mach_vals = [engine.V_obs[j], engine.I_obs[j], res['v_corr'][j], res['Q_eff'][j]]
    mach_refs = [engine.V0, engine.I0, engine.v0, engine.target_Q]
    mach_ratios_all[:, j] = [100 + 10 * (v - ref) / tol for v, ref, tol in zip(mach_vals, mach_refs, TOL_MACH)]

env_bar_max = max(180, np.max(env_ratios_all) * 1.25) 
mach_bar_max = max(150, np.max(mach_ratios_all) * 1.55)

AXIS_LIMITS = {
    'spatial_x': (0, MAX_X_POS), 'spatial_y': (0, 25),
    'cross_x': (-10, 10), 'cross_y': (-10, 10),
    'env_bar_y': (0, env_bar_max),
    'phase_I': (170, 230), 'phase_V': (21.5, 28.5),
    'mach_bar_y': (0, mach_bar_max),
    'Q_y': (engine.target_Q * 0.6, engine.target_Q * 1.4),
    'v_y': (4, 16), 
    'F_y': (0, max(res['F_pull']) * 1.2),
    'P_y': (0, 110) 
}

# 공통 컬러 및 스타일 설정
C_SAFE, C_CAUT, C_RISK = '#d9f0d3', '#fdf0c3', '#f4cbcb'
FG_SAFE, FG_CAUT, FG_RISK = '#2ca02c', '#ff9800', '#d62728'
BG_ALPHA = 0.6
MARKER_STYLE = {'marker': 'o', 'markersize': 9, 'markeredgecolor': 'black', 'zorder': 5}
LINE_STYLE = {'linewidth': 1.25, 'marker': 'o', 'markerfacecolor': 'none', 'markersize': 5, 'markeredgewidth': 1.2, 'zorder': 3}
LINE_STYLE_THIN = {'linewidth': 0.625, 'marker': 'o', 'markerfacecolor': 'none', 'markersize': 5, 'markeredgewidth': 1.2, 'zorder': 3}

# ----------------- ROW 1 -----------------
# 1. 선형 토치 뷰 
ax_spatial = axes[0, 0]
ax_spatial.set_title("1. Linear Torch & Bead Cooling")
ax_spatial.set_xlim(*AXIS_LIMITS['spatial_x']); ax_spatial.set_ylim(*AXIS_LIMITS['spatial_y'])
ax_spatial.set_xlabel("Weld Position X (mm)"); ax_spatial.set_ylabel("Z Height (mm)")
ax_spatial.fill_between(res['x_pos'], 0, engine.Tp_obs, color='slategray', alpha=0.5, zorder=1)

ax_spatial.legend(handles=[
    mpatches.Patch(color='green', label='Torch'),
    mpatches.Patch(color='slategray', alpha=0.5, label='Base Metal')
], loc='upper right', framealpha=0.8)

torch_line, = ax_spatial.plot([], [], 'g-', linewidth=6, solid_capstyle='round', zorder=3)
arc_glow, = ax_spatial.plot([], [], 'o', color='yellow', markersize=15, alpha=0.6, zorder=2)

min_temp = 25.0
max_temp = 1800.0
bead_scatter = ax_spatial.scatter([], [], c=[], cmap='hot', s=60, vmin=min_temp, vmax=max_temp, edgecolors='none', zorder=2)

cbar = fig.colorbar(bead_scatter, ax=ax_spatial, fraction=0.046, pad=0.04)
cbar.set_label("Weld Pool Temp (°C)", fontweight='bold')

# 2. 비드 & 너겟 단면 뷰
ax_cross = axes[0, 1]
ax_cross.set_title("2. Bead & Nugget Cross-section")
ax_cross.set_xlim(*AXIS_LIMITS['cross_x']); ax_cross.set_ylim(*AXIS_LIMITS['cross_y']); ax_cross.set_aspect('equal')
ax_cross.set_xlabel("Transverse Width (mm)"); ax_cross.set_ylabel("Depth & Height (mm)")
base_metal_bot, = ax_cross.plot([-10, 10], [-engine.Tp0, -engine.Tp0], 'k--', linewidth=2, zorder=1)
ax_cross.axhline(0, color='black', linewidth=2, zorder=1)

bead_outlines = [Ellipse((0, 0), width=0, height=0, angle=0, fill=False, edgecolor='#ff7f0e', linestyle=':', alpha=0.2, zorder=1, visible=False) for _ in range(engine.steps)]
nugget_outlines = [Ellipse((0, -engine.Tp0/2), width=0, height=0, angle=0, fill=False, edgecolor='#d62728', linestyle=':', alpha=0.2, zorder=1, visible=False) for _ in range(engine.steps)]
for bo, no in zip(bead_outlines, nugget_outlines):
    ax_cross.add_patch(bo)
    ax_cross.add_patch(no)

bead_patch = Ellipse((0, 0), width=0, height=0, angle=0, color='#ff7f0e', alpha=0.6, zorder=2)
nugget_patch = Ellipse((0, -engine.Tp0/2), width=0, height=0, angle=0, color='#d62728', alpha=0.9, zorder=3)
ax_cross.add_patch(bead_patch); ax_cross.add_patch(nugget_patch)
ax_cross.legend(handles=[
    mpatches.Patch(color='#ff7f0e', alpha=0.6, label='Bead Area'),
    mpatches.Patch(color='#d62728', alpha=0.9, label='Nugget Area')
], loc='upper right')

# 3. 환경 변수 막대 그래프 
ax_env_bar = axes[0, 2]
ax_env_bar.set_title("3. Environment & Shape (vs Standard)")
env_labels = ['Temp\n(25°C)', 'Angle\n(15°)', 'Depth\n(Target)', 'Width\n(Target)']
env_bars = ax_env_bar.bar(env_labels, [100]*4, width=0.64, zorder=2) 
ax_env_bar.axhline(100, color='red', linestyle='--', label='Standard (100%)', zorder=3)
ax_env_bar.set_ylim(*AXIS_LIMITS['env_bar_y']); ax_env_bar.set_ylabel("Normalized Tolerance Index (%)")
ax_env_bar.axhspan(90, 110, color=C_SAFE, alpha=BG_ALPHA, zorder=0)
ax_env_bar.axhspan(75, 90, color=C_CAUT, alpha=BG_ALPHA, zorder=0)
ax_env_bar.axhspan(110, 125, color=C_CAUT, alpha=BG_ALPHA, zorder=0)
ax_env_bar.axhspan(-1000, 75, color=C_RISK, alpha=BG_ALPHA, zorder=0)
ax_env_bar.axhspan(125, 10000, color=C_RISK, alpha=BG_ALPHA, zorder=0) 
env_texts = [ax_env_bar.text(i, 110, '', ha='center', fontweight='bold', fontsize=12, zorder=3) for i in range(4)]

env_leg = ax_env_bar.legend(handles=[
    mpatches.Patch(color=C_SAFE, alpha=BG_ALPHA, label='Safe Range'),
    mpatches.Patch(color=C_CAUT, alpha=BG_ALPHA, label='Caution Range'),
    mpatches.Patch(color=C_RISK, alpha=BG_ALPHA, label='Risk Range'),
    plt.Line2D([0], [0], color='red', linestyle='--', lw=1.5, label='Standard (100%)')
], loc='upper right', framealpha=0.5, facecolor='white')
env_leg.set_zorder(6)

env_err_lines = [ax_env_bar.plot([], [], 'k-', lw=1.5, zorder=4)[0] for _ in range(4)]
env_err_caps_top = [ax_env_bar.plot([], [], 'k_', markersize=10, markeredgewidth=1.5, zorder=4)[0] for _ in range(4)]
env_err_caps_bot = [ax_env_bar.plot([], [], 'k_', markersize=10, markeredgewidth=1.5, zorder=4)[0] for _ in range(4)]

# ----------------- ROW 2 -----------------
# 4. V-I 상도표 
ax_phase = axes[1, 0]
ax_phase.set_title("4. V-I Phase Diagram")
I_mesh, V_mesh = np.meshgrid(np.linspace(*AXIS_LIMITS['phase_I'], 100), np.linspace(*AXIS_LIMITS['phase_V'], 100))
Q_mesh = (engine.eta * V_mesh * I_mesh) / engine.v0
Q_safe_lo, Q_safe_hi = engine.target_Q * 0.95, engine.target_Q * 1.05
Q_warn_lo, Q_warn_hi = engine.target_Q * 0.85, engine.target_Q * 1.15
levels = [0, Q_warn_lo, Q_safe_lo, Q_safe_hi, Q_warn_hi, 1000]
colors = [C_RISK, C_CAUT, C_SAFE, C_CAUT, C_RISK] 
cf = ax_phase.contourf(I_mesh, V_mesh, Q_mesh, levels=levels, colors=colors, alpha=BG_ALPHA, zorder=0)
ax_phase.set_xlabel('Current $I$ (A)'); ax_phase.set_ylabel('Voltage $V$ (V)')

phase_leg = ax_phase.legend(handles=[
    mpatches.Patch(color=C_SAFE, alpha=BG_ALPHA, label='Safe Phase'),
    mpatches.Patch(color=C_CAUT, alpha=BG_ALPHA, label='Caution Phase'),
    mpatches.Patch(color=C_RISK, alpha=BG_ALPHA, label='Risk Phase')
], loc='upper right', framealpha=0.5, facecolor='white')
phase_leg.set_zorder(6)

phase_tail, = ax_phase.plot([], [], color='gray', linestyle='-', linewidth=0.75, marker='o', markerfacecolor='none', markeredgecolor='gray', markersize=4, alpha=0.6, zorder=2)
phase_point, = ax_phase.plot([], [], linestyle='None', **MARKER_STYLE)
ax_phase.density_contour = None 

# 5. 유효 열입력
ax_Q = axes[1, 1]
ax_Q.set_title("5. Effective Heat Input $Q_{eff}(t)$")
ax_Q.set_xlim(0, MAX_TIME); ax_Q.set_ylim(*AXIS_LIMITS['Q_y'])
ax_Q.set_xlabel("Time (s)"); ax_Q.set_ylabel("Heat Input (J/mm)")
ax_Q.axhline(engine.target_Q, color='gray', linestyle='--', linewidth=2, zorder=2)
ax_Q.axhspan(Q_safe_lo, Q_safe_hi, color=C_SAFE, alpha=BG_ALPHA, zorder=0)
ax_Q.axhspan(Q_warn_lo, Q_safe_lo, color=C_CAUT, alpha=BG_ALPHA, zorder=0)
ax_Q.axhspan(Q_safe_hi, Q_warn_hi, color=C_CAUT, alpha=BG_ALPHA, zorder=0)
ax_Q.axhspan(-1000, Q_warn_lo, color=C_RISK, alpha=BG_ALPHA, zorder=0)
ax_Q.axhspan(Q_warn_hi, 10000, color=C_RISK, alpha=BG_ALPHA, zorder=0)
line_Q, = ax_Q.plot([], [], color='black', label='Effective Heat Input', **LINE_STYLE_THIN)
ax_Q.legend(loc='upper right')
Q_point, = ax_Q.plot([], [], linestyle='None', **MARKER_STYLE)

# 6. 장비 파라미터 막대 그래프
ax_mach_bar = axes[1, 2]
ax_mach_bar.set_title("6. Machine Parameters (vs Standard)")
mach_labels = ['Voltage\n(24V)', 'Current\n(200A)', 'Speed\n(8mm/s)', 'Heat Input\n(Target)']
mach_bars = ax_mach_bar.bar(mach_labels, [100]*4, width=0.64, zorder=2) 
ax_mach_bar.axhline(100, color='red', linestyle='--', label='Standard (100%)', zorder=3)
ax_mach_bar.set_ylim(*AXIS_LIMITS['mach_bar_y']); ax_mach_bar.set_ylabel("Normalized Tolerance Index (%)")
ax_mach_bar.axhspan(90, 110, color=C_SAFE, alpha=BG_ALPHA, zorder=0)
ax_mach_bar.axhspan(75, 90, color=C_CAUT, alpha=BG_ALPHA, zorder=0)
ax_mach_bar.axhspan(110, 125, color=C_CAUT, alpha=BG_ALPHA, zorder=0)
ax_mach_bar.axhspan(-1000, 75, color=C_RISK, alpha=BG_ALPHA, zorder=0)
ax_mach_bar.axhspan(125, 10000, color=C_RISK, alpha=BG_ALPHA, zorder=0)
mach_texts = [ax_mach_bar.text(i, 110, '', ha='center', fontweight='bold', fontsize=12, zorder=3) for i in range(4)]

mach_leg = ax_mach_bar.legend(handles=[
    mpatches.Patch(color=C_SAFE, alpha=BG_ALPHA, label='Safe Range'),
    mpatches.Patch(color=C_CAUT, alpha=BG_ALPHA, label='Caution Range'),
    mpatches.Patch(color=C_RISK, alpha=BG_ALPHA, label='Risk Range'),
    plt.Line2D([0], [0], color='red', linestyle='--', lw=1.5, label='Standard (100%)')
], loc='upper right', framealpha=0.5, facecolor='white')
mach_leg.set_zorder(6)

mach_err_lines = [ax_mach_bar.plot([], [], 'k-', lw=1.5, zorder=4)[0] for _ in range(4)]
mach_err_caps_top = [ax_mach_bar.plot([], [], 'k_', markersize=10, markeredgewidth=1.5, zorder=4)[0] for _ in range(4)]
mach_err_caps_bot = [ax_mach_bar.plot([], [], 'k_', markersize=10, markeredgewidth=1.5, zorder=4)[0] for _ in range(4)]

# ----------------- ROW 3 -----------------
# 7. 토치 속도 조절
ax_v = axes[2, 0]
ax_v.set_title("7. Torch Speed Adjustment $v(t)$")
ax_v.set_xlim(0, MAX_TIME); ax_v.set_ylim(*AXIS_LIMITS['v_y'])
ax_v.set_xlabel("Time (s)"); ax_v.set_ylabel("Torch Speed (mm/s)")
ax_v.axhline(engine.v0, color='gray', linestyle='--', linewidth=2, label='Nominal Speed', zorder=2)
ax_v.axhspan(7.5, 8.5, color=C_SAFE, alpha=BG_ALPHA, zorder=0)
ax_v.axhspan(6.5, 7.5, color=C_CAUT, alpha=BG_ALPHA, zorder=0)
ax_v.axhspan(8.5, 9.5, color=C_CAUT, alpha=BG_ALPHA, zorder=0)
ax_v.axhspan(-1000, 6.5, color=C_RISK, alpha=BG_ALPHA, zorder=0)
ax_v.axhspan(9.5, 10000, color=C_RISK, alpha=BG_ALPHA, zorder=0)
line_v, = ax_v.plot([], [], color='blue', label='AI Corrected Speed', **LINE_STYLE_THIN)
ax_v.legend(loc="upper right")
v_point, = ax_v.plot([], [], linestyle='None', **MARKER_STYLE)

# 8. 인장 전단 강도
ax_F = axes[2, 1]
ax_F.set_title("8. Tensile Shear Strength $F_{pull}(t)$")
ax_F.set_xlim(0, MAX_TIME); ax_F.set_ylim(*AXIS_LIMITS['F_y'])
ax_F.set_xlabel("Time (s)"); ax_F.set_ylabel("Tensile Strength (kN)")
ax_F.axhspan(80, 10000, color=C_SAFE, alpha=BG_ALPHA, zorder=0)
ax_F.axhspan(60, 80, color=C_CAUT, alpha=BG_ALPHA, zorder=0)
ax_F.axhspan(-1000, 60, color=C_RISK, alpha=BG_ALPHA, zorder=0)
line_F, = ax_F.plot([], [], color='magenta', label='Tensile Strength', **LINE_STYLE_THIN)
ax_F.legend(loc='upper right')
F_point, = ax_F.plot([], [], linestyle='None', **MARKER_STYLE)

# 9. 결함 확률
ax_P = axes[2, 2]
ax_P.set_title("9. Defect Probability & Guardrail")
ax_P.set_xlim(0, MAX_TIME); ax_P.set_ylim(*AXIS_LIMITS['P_y'])
ax_P.set_xlabel("Time (s)"); ax_P.set_ylabel("Defect Probability (%)")
ax_P.axhline(100, color='red', linestyle='--', linewidth=1.5, zorder=2) 
ax_P.axhspan(0, 20, color=C_SAFE, alpha=BG_ALPHA, zorder=0)
ax_P.axhspan(20, 50, color=C_CAUT, alpha=BG_ALPHA, zorder=0)
ax_P.axhspan(50, 100, color=C_RISK, alpha=BG_ALPHA, zorder=0)
line_P, = ax_P.plot([], [], color='red', label='Defect Probability', **LINE_STYLE)
ax_P.legend(loc='upper right')
P_point, = ax_P.plot([], [], linestyle='None', **MARKER_STYLE)

# ------------------------------------------
# 상태 판별 헬퍼 함수
# ------------------------------------------
def get_ratio_color(r):
    if r < 75 or r > 125: return FG_RISK
    if r < 90 or r > 110: return FG_CAUT
    return FG_SAFE
def get_Q_color(q):
    if q < Q_warn_lo or q > Q_warn_hi: return FG_RISK
    if q < Q_safe_lo or q > Q_safe_hi: return FG_CAUT
    return FG_SAFE
def get_v_color(v):
    if v < 6.5 or v > 9.5: return FG_RISK
    if v < 7.5 or v > 8.5: return FG_CAUT
    return FG_SAFE
def get_F_color(f):
    if f < 60: return FG_RISK
    if f < 80: return FG_CAUT
    return FG_SAFE
def get_P_color(p):
    if p > 50: return FG_RISK
    if p > 20: return FG_CAUT
    return FG_SAFE

# ------------------------------------------
# 애니메이션 렌더링 로직
# ------------------------------------------
history_x, history_y, history_temp = [], [], []

def update(frame):
    curr_t = engine.time[frame]
    q_val = res['Q_eff'][frame]
    
    # 1. Spatial
    cx, z = res['x_pos'][frame], engine.Tp_obs[frame]
    rad = np.radians(engine.theta_eff[frame])
    tx, tz = cx - engine.dist_eff[frame] * np.sin(rad), z + engine.dist_eff[frame] * np.cos(rad)
    torch_line.set_data([cx, tx], [z, tz]); arc_glow.set_data([cx], [z + 1])
    
    melt_temp = 1500.0 + (q_val - engine.target_Q) * 2.0
    history_x.append(cx); history_y.append(z); history_temp.append(melt_temp)
    for i in range(len(history_temp)): 
        history_temp[i] = max(min_temp, history_temp[i] * 0.92 - 10.0)
        
    bead_scatter.set_offsets(np.column_stack((history_x, history_y)))
    bead_scatter.set_array(np.array(history_temp))
    
    # 2. Cross-section
    w, d = res['W'][frame], res['D'][frame]
    bead_patch.width, bead_patch.height = w, d * 2
    nugget_patch.width, nugget_patch.height = res['d_nugget'][frame], res['d_nugget'][frame] * 0.6
    nugget_patch.center = (0, -engine.Tp_obs[frame]/2)
    base_metal_bot.set_data([-10, 10], [-engine.Tp_obs[frame], -engine.Tp_obs[frame]])
    
    bead_outlines[frame].width, bead_outlines[frame].height = w, d * 2
    bead_outlines[frame].set_visible(True)
    nugget_outlines[frame].width, nugget_outlines[frame].height = res['d_nugget'][frame], res['d_nugget'][frame] * 0.6
    nugget_outlines[frame].center = (0, -engine.Tp_obs[frame]/2)
    nugget_outlines[frame].set_visible(True)
    
    # 3. Environment Bar Chart
    env_vals = [engine.T_surf[frame], engine.theta_eff[frame], d, w]
    env_refs = [engine.T0, engine.theta0, target_D, target_W]
    env_ratios = env_ratios_all[:, frame]
    formats = ["{:.1f} °C", "{:.1f} °", "{:.2f} mm", "{:.2f} mm"]
    
    for i, (bar, ratio, val) in enumerate(zip(env_bars, env_ratios, env_vals)):
        bar.set_height(ratio)
        bar.set_color(get_ratio_color(ratio))
        env_texts[i].set_text(formats[i].format(val))
        env_texts[i].set_position((i, ratio + (env_bar_max * 0.025)))
        e_min = np.min(env_ratios_all[i, :frame+1])
        e_max = np.max(env_ratios_all[i, :frame+1])
        env_err_lines[i].set_data([i, i], [e_min, e_max])
        env_err_caps_top[i].set_data([i], [e_max])
        env_err_caps_bot[i].set_data([i], [e_min])
        
    # 4. Phase Diagram 
    phase_tail.set_data(engine.I_obs[:frame], engine.V_obs[:frame]) 
    phase_point.set_data([engine.I_obs[frame]], [engine.V_obs[frame]])
    phase_point.set_markerfacecolor(get_Q_color(q_val))
    
    if hasattr(ax_phase, 'density_contour') and ax_phase.density_contour:
        try:
            ax_phase.density_contour.remove()
        except ValueError:
            pass 
            
    if frame > 5: 
        I_data = engine.I_obs[:frame+1]
        V_data = engine.V_obs[:frame+1]
        H, xedges, yedges = np.histogram2d(I_data, V_data, bins=15, 
                                           range=[AXIS_LIMITS['phase_I'], AXIS_LIMITS['phase_V']])
        H = ndimage.gaussian_filter(H, sigma=1.0)
        X, Y = np.meshgrid(xedges[:-1] + np.diff(xedges)/2, yedges[:-1] + np.diff(yedges)/2)
        
        if np.max(H) > 0:
            ax_phase.density_contour = ax_phase.contour(X, Y, H.T, levels=3, colors='black', linestyles='dashed', alpha=0.5, linewidths=1.0, zorder=1)

    # 5. Q(t) (Line)
    t = engine.time[:frame+1]
    line_Q.set_data(t, res['Q_eff'][:frame+1])
    Q_point.set_data([curr_t], [q_val])
    Q_point.set_markerfacecolor(get_Q_color(q_val))

    # 6. Machine Bar Chart
    mach_vals = [engine.V_obs[frame], engine.I_obs[frame], res['v_corr'][frame], q_val]
    mach_ratios = mach_ratios_all[:, frame]
    formats_m = ["{:.1f} V", "{:.1f} A", "{:.1f} mm/s", "{:.0f} J"]
    
    for i, (bar, ratio, val) in enumerate(zip(mach_bars, mach_ratios, mach_vals)):
        bar.set_height(ratio)
        bar.set_color(get_ratio_color(ratio))
        mach_texts[i].set_text(formats_m[i].format(val))
        mach_texts[i].set_position((i, ratio + (mach_bar_max * 0.025)))
        m_min = np.min(mach_ratios_all[i, :frame+1])
        m_max = np.max(mach_ratios_all[i, :frame+1])
        mach_err_lines[i].set_data([i, i], [m_min, m_max])
        mach_err_caps_top[i].set_data([i], [m_max])
        mach_err_caps_bot[i].set_data([i], [m_min])
        
    # 7~9. Lines
    v_val = res['v_corr'][frame]
    line_v.set_data(t, res['v_corr'][:frame+1])
    v_point.set_data([curr_t], [v_val])
    v_point.set_markerfacecolor(get_v_color(v_val))
    
    F_val = res['F_pull'][frame]
    line_F.set_data(t, res['F_pull'][:frame+1])
    F_point.set_data([curr_t], [F_val])
    F_point.set_markerfacecolor(get_F_color(F_val))
    
    P_val = res['P_defect'][frame] * 100
    line_P.set_data(t, res['P_defect'][:frame+1] * 100)
    P_point.set_data([curr_t], [P_val])
    P_point.set_markerfacecolor(get_P_color(P_val))
        
    return (torch_line, arc_glow, bead_scatter, bead_patch, nugget_patch, base_metal_bot, 
            phase_point, phase_tail, line_Q, Q_point, line_v, v_point, line_F, F_point, line_P, P_point, 
            *env_bars, *env_texts, *mach_bars, *mach_texts, 
            *env_err_lines, *env_err_caps_top, *env_err_caps_bot, 
            *mach_err_lines, *mach_err_caps_top, *mach_err_caps_bot,
            *bead_outlines, *nugget_outlines)

anim = animation.FuncAnimation(fig, update, frames=engine.steps, interval=FRAME_INTERVAL, blit=False)
plt.close(fig)

gif_path = os.path.join(RESULT_DIR, "3x3_ultimate_dashboard_normalized.gif")
anim.save(gif_path, writer='pillow', fps=ANIMATION_FPS)
print(f"✅ [완료] 슬라이더 제거 및 그래프 여백 최적화가 완벽하게 적용된 최종본 저장 완료: {gif_path}")

HTML(anim.to_jshtml())

## 5. 적응형 AI 제어 기반 용접 시뮬레이션 결과 분석

본 디지털 트윈 대시보드는 30초의 공정 구간 동안 발생하는 다중 물리 외란(Multi-physics Disturbance)에 대해 AI 제어기가 수행하는 실시간 보상 제어(Closed-loop Compensation)의 실제 거동을, 렌더링된 시뮬레이션 결과물(`result_sim/3x3_ultimate_dashboard_normalized.gif`)의 초기·종료 프레임 검토에 근거하여 분석한다. 3절에서 명시한 바와 같이 본 제어기의 조작 변수는 토치 속도 $v$ 단일 변수로 한정되므로, 이하 분석은 **제어 대상인 유효 열입력 계열 지표**와 **제어 범위 밖에 있는 자유 확산 외란 계열 지표**를 구분하여 기술한다.

---

### 1. 실시간 공정 시각화 및 형상 계측(상단 패널)

* **[1행 1열] 선형 궤적 및 용융풀 냉각 모델:** 토치(녹색 선)의 종방향 진동은 팁-모재 간 거리(CTWD)의 물리적 노이즈 편차를 반영한다. 컬러맵은 열역학적 상변화를 모사하며, 최고 1,800°C에 근접하는 용융 금속이 상온(25°C)으로 급속 냉각(Quenching)되는 열 이력을 나타낸다.

* **[1행 2열] 접합 단면 기하학:** 비드(주황색)와 너겟(적색)의 면적은 용입 건전성을 대변한다. 배경에 중첩된 점선 궤적은 300개 프레임 전체에 걸쳐 누적된 단면 앙상블(Ensemble)이며, 그 포락선(Envelope)의 폭이 비교적 좁게 유지되는 것으로 보아 — 후술할 유효 열입력의 안정적 수렴에 기인하여 — 형상 재현성이 상당히 양호하게 유지됨을 확인할 수 있다.

* **[1행 3열] 환경 및 기하 치수 응답:** 공정 종료 시점(t=30s) 기준으로 형상 관련 지수(Depth, Width)는 정규화 지수 100% 부근의 안전(녹색) 영역에 안정적으로 유지된다. 그러나 **표면온도(Temp)와 접촉각(Angle) 지수는 각각 약 265%, 127% 수준까지 상승하여 명백한 위험(적색) 영역에 도달**한다. 이는 온도·각도 편차 자체가 제어기의 직접적 보정 대상이 아니라 자유롭게 누적되는 외란임을 시사하며, 제어기는 그 파급 효과인 형상 지수만을 목표 범위 내로 억제하고 있음을 보여준다.

---

### 2. 전자기 위상 공간 및 제어 에너지 추적(중단 패널)

* **[2행 1열] V-I 위상 다이어그램(Phase Diagram):** 전류-전압 동작점의 시간 궤적(회색 산점)은 전원 장치 고유의 랜덤워크성 노이즈에 의해 자유롭게 표류하며, 공정 후반부에는 궤적이 안전(녹색) 대각선 띠를 벗어나 경계(황색) 및 위험(적색) 배경 구간까지 반복적으로 진입하는 양상을 보인다. 배경의 점선 등고선은 해당 동작점 분포의 커널 밀도 추정(KDE) 결과이다. 즉 $V$, $I$ 자체는 제어기의 조작 대상이 아니므로 위상 공간상에서 자유 확산(Free Diffusion)하며, 이는 후술하는 결함 확률 변동성과 연동되는 주요 원인 중 하나이다.

* **[2행 2열] 실시간 유효 열입력 $Q_{eff}(t)$:** 본 제어 루프의 **직접적 제어 대상**이다. 공정 초반 약 5초 구간에서는 큰 진폭의 과도 응답(Transient Oscillation)이 관측되나, 이후 제어기의 개입으로 목표 밴드(목표값 ± 안전 마진) 중앙부에 즉시 흡착(Clamping)되어 잔여 25초 구간 동안 매우 낮은 분산으로 안정적으로 유지된다. 이는 단일변수 보상 제어가 설계 의도대로 작동함을 입증하는 가장 직접적인 증거이다.

* **[2행 3열] 기계 제어 출력 상태:** 공정 종료 시점 스냅샷에서 전압(Voltage)과 속도(Speed) 지수는 위험(적색) 영역에, 전류(Current)와 열입력(Heat Input) 지수는 안전(녹색) 영역에 위치한다. 속도가 위험 영역에 도달한 것은 제어 실패가 아니라, 누적된 외란을 상쇄하기 위해 조작 변수가 명목값(8 mm/s) 대비 약 46% 증가한 결과로 해석해야 하며, 오히려 열입력이 안전하게 유지되고 있다는 사실이 이 보상이 유효하게 작동하고 있음을 뒷받침한다.

---

### 3. 제어기 응답 성능 및 기계적 품질 평가(하단 패널)

* **[3행 1열] 제어 조작 변수(MV) 동적 응답 $v(t)$:** AI 제어기의 핵심 보상 기제(Compensation Mechanism)를 직접 보여주는 지표이다. 공정 전반부에는 명목 속도(8 mm/s) 부근에서 소폭 진동하나, 후반부(t≈25–30s)로 갈수록 누적 외란에 대응하여 속도가 지속적으로 상향 조정되며 공정 종료 시점에는 약 12 mm/s까지 상승한다. 이는 제어기가 정적(Static) 보상이 아니라 누적 외란의 크기에 비례하여 개입 강도를 증가시키는 동적(Dynamic) 보상을 수행하고 있음을 의미한다.

* **[3행 2열] 인장 전단 강도 $F_{pull}(t)$:** 시뮬레이션 결과, 해당 지표는 60–100 kN 부근의 준수한 값과 0 kN에 근접하는 급락을 반복하는 **고빈도·고진폭의 비정상(Erratic) 거동**을 보인다. 이는 $F_{pull}$이 결함 확률 $P_{defect}$의 감소함수($F_{pull}\propto(1-P_{defect})$)로 정의되어 있어, 아래 항에서 기술하는 결함 확률의 진동이 그대로 전이된 결과이다. 즉 이 패널은 접합부 강도가 "안정적으로 유지"된다기보다, **결함 확률 변동성에 종속되어 함께 요동하는 하위 지표**로 해석하는 것이 타당하다.

* **[3행 3열] 결함 발생 확률(Defect Probability) 및 가드레일:** 결함 확률 $P_{defect}$는 유효 열입력 편차 항과 접촉각 초과분 항의 가중합을 로지스틱 함수에 대입하여 산출된다($risk\_score = 0.1|Q_{eff}-Q_0| + 0.5\max(0,\theta_{eff}-15°)$). 유효 열입력 항은 제어기에 의해 즉시 0에 수렴하지만, **접촉각 항은 제어기가 개입할 수 있는 조작 변수가 존재하지 않으므로 접촉각의 랜덤워크성 누적 편차가 그대로 결함 확률에 전이된다.** 그 결과 공정 전 구간에 걸쳐 결함 확률은 0%와 100% 사이를 고빈도로 진동하며, **"0%로 수렴·고정"되는 이상적 거동은 관측되지 않는다.** 이는 결함 방지 성능의 실패라기보다, 본 시뮬레이션이 구현한 가드레일이 **열입력 단일 축에 대한 부분적(Partial) 보상 체계**이며 접촉각(굴곡) 축에 대해서는 개루프(Open-loop) 상태로 방치되어 있다는 **설계상의 한계**를 정량적으로 드러내는 결과로 해석해야 한다.

**[정리]**

본 시뮬레이션 대시보드는 **외란 인입 → 상태 변화 감지 → 조작 변수(토치 속도)의 보상 → 유효 열입력 안정화**로 이어지는 폐루프 제어의 절반, 즉 **열입력 축에 한정된 단일변수 제어의 공학적 타당성**을 명확히 검증한다. 그러나 결함 확률 및 이에 종속된 인장 전단 강도 지표는 열입력 제어만으로는 안정화되지 않으며, 이는 접촉각(굴곡) 편차에 대한 별도의 보상 축이 부재하다는 시스템 설계상의 근본적 한계에 기인한다. 이 한계에 대한 종합적 논의는 다음 절(요약)에서 이어간다.

## 6. 패널별 결과 요약 및 종합 결론 (Panel-wise Result Synthesis)

`result_sim/3x3_ultimate_dashboard_normalized.gif`의 300프레임(30초, FPS=10) 전 구간 중 초기·종료 프레임을 직접 디코딩하여 검토한 결과를 9개 패널별로 정리한다. 각 항목은 **한줄 요약**과 이를 뒷받침하는 상세 설명으로 구성한다.

**1. 선형 토치 궤적 및 냉각 모델 — 한줄 요약:** 토치는 팁-모재 거리 노이즈에 의해 미세하게 진동하며, 궤적을 따라 생성된 용융풀은 매 프레임 즉시 냉각·응고된다.  
　상세: 산점도의 컬러맵(`hot`, 25–1,800°C)은 매 프레임 신규 생성되는 용융점을 최고 온도로 표시한 뒤, 이후 프레임마다 지수적 감쇠식($T_i \leftarrow \max(25,\ 0.92\,T_i - 10)$)을 적용해 급속 냉각을 근사적으로 모사한다. 이 냉각 모델은 실측 열전달 방정식이 아니라 시각적 렌더링을 위한 경험적 근사임에 유의해야 한다.

**2. 비드 및 너겟 단면 — 한줄 요약:** 단면 형상은 300프레임 앙상블 전체에서 뚜렷한 크기 변동 없이 높은 재현성을 유지한다.  
　상세: 비드 폭 $W=0.4\sqrt{Q_{eff}}$, 깊이 $D=0.04\,Q_{eff}^{0.8}$이 모두 유효 열입력의 함수로 정의되어 있어, $Q_{eff}$가 제어기에 의해 좁은 밴드 내로 고정되는 즉시 형상도 함께 고정된다. 즉 이 패널에서 관측되는 낮은 분산은 형상 자체의 독립적인 물리적 강건성이 아니라, 상위 지표인 $Q_{eff}$가 이미 안정화된 결과의 재확인(Corollary)으로 해석해야 한다.

**3. 환경 및 형상 정규화 막대 — 한줄 요약:** 형상 지수(Depth·Width)는 안전 영역을 유지하는 반면, 환경 지수(Temp·Angle)는 공정 종료 시점 위험 영역까지 이탈한다.  
　상세: t=30s 시점에서 Temp 지수는 약 265%(실측 108.3°C, 목표 25°C), Angle 지수는 약 127%(실측 22.9°, 목표 15°)로 관측되어 위험(적색) 범주에 해당한다. 반면 Depth·Width 지수는 각각 100% 부근의 안전(녹색) 범주를 유지한다. 이는 제어기가 온도·각도라는 "원인 변수"가 아니라 그로부터 파생되는 "결과 변수(형상)"만을 목표로 보정하고 있다는 3절의 설계 범위 한정과 정확히 부합하는 관측이다.

**4. V-I 위상 다이어그램 — 한줄 요약:** 전류-전압 동작점은 뚜렷한 제어 없이 위상 공간 내에서 자유롭게 확산하며 위험 구역 진입을 배제하지 못한다.  
　상세: $I$, $V$는 각각 독립적인 누적 랜덤워크(`cumsum(N(0,0.8))`, `cumsum(N(0,0.1))`)로 생성되며 제어 루프의 피드백을 받지 않는다. 공정 후반 동작점 궤적(회색 점선)은 안전(녹색) 대각선 띠를 벗어나 경계 및 위험 배경까지 반복적으로 진입하며, 배경 등고선(KDE)이 넓게 퍼진 형태로 이 확산성을 뒷받침한다.

**5. 유효 열입력 $Q_{eff}(t)$ — 한줄 요약:** 초기 약 5초의 과도 구간을 지나면 유효 열입력은 목표값에 즉시 고정되어 잔여 구간 내내 안정적으로 유지된다.  
　상세: 이는 본 제어 시스템에서 유일하게 명시적인 피드백 보정이 적용되는 지표로, 실측 결과 역시 설계 의도(목표 밴드 중심 유지)를 가장 충실히 재현한다. 관측된 낮은 분산은 2·3·6번 패널의 안정적 결과를 설명하는 근본 원인이다.

**6. 장비 제어 변수 정규화 막대 — 한줄 요약:** 전압은 외란에 노출되어 위험 영역에 도달하는 반면, 속도는 큰 폭의 능동적 보상값을 나타내면서도 결과적으로 열입력만은 안전 영역에 묶어 둔다.  
　상세: t=30s 스냅샷에서 Voltage 지수 약 127%(위험), Current 지수 약 103%(안전이나 우연적 스냅샷), Speed 지수 약 137%(위험으로 표시되나 이는 능동 보상의 크기를 반영), Heat Input 지수 약 100%(안전)로 나타난다. Speed의 "위험" 표시는 제어 실패가 아니라 보상 강도 자체가 크다는 의미로 해석해야 하며, 최종 목적 변수인 Heat Input이 안전하다는 점이 이 보상이 유효하게 작동하고 있다는 실질적 근거이다.

**7. 조작 변수 제어 속도 $v(t)$ — 한줄 요약:** 토치 속도는 공정 진행에 따라 누적 외란에 비례하여 점진적으로 증가하는 동적 보상 궤적을 나타낸다.  
　상세: 초반 약 3초는 명목 속도(8 mm/s) 근방에서 유지되다가 이후 소폭 진동을 동반하며 상승 추세를 보이고, 특히 t≈25–30s 구간에서 급격히 상승하여 약 12 mm/s(명목 대비 +46%)에 도달한다. 이는 외란의 누적 효과가 시간에 따라 커지고 있으며, 제어기가 이에 비례하여 개입 강도를 강화하고 있음을 시사한다.

**8. 인장 전단 강도 $F_{pull}(t)$ — 한줄 요약:** 접합부 강도 추정치는 안정적 수렴 없이 0 kN 근방과 90–100 kN 근방을 반복적으로 오가는 고진폭 진동을 나타낸다.  
　상세: $F_{pull}\propto d_{nugget}^2\times(1-P_{defect})$로 정의되어 결함 확률의 함수로 종속되므로, 9번 항목에서 기술하는 결함 확률의 진동이 그대로 강도 추정치에 전이된다. 결과적으로 "안전 대역 유지"라는 이상적 서술과 달리, 실측 궤적은 안전(녹색)과 위험(적색) 영역을 초당 수 회 수준의 빈도로 넘나든다.

**9. 결함 발생 확률 및 가드레일 — 한줄 요약:** 결함 확률은 0%로 수렴·고정되지 않고 공정 전 구간에서 0–100% 사이를 고빈도로 진동한다.  
　상세: 결함 확률 산식의 두 항 중 열입력 항은 제어기에 의해 신속히 0에 근접하지만, 접촉각 초과항($0.5\max(0,\theta_{eff}-15°)$)은 별도의 보정 구동기가 없는 순수 랜덤워크 변수에 의해 좌우된다. 접촉각의 표준편차는 시간에 비례하여 증가하는 비정상 확률과정이므로 15° 임계값을 빈번히 상·하로 교차하며, 이 교차마다 결함 확률이 0%와 100% 사이를 오가는 급격한 스파이크를 유발한다. 이는 본 시뮬레이션이 제시하는 가드레일이 완전한(Full) 다변수 보상 체계가 아니라 **열입력 단일 축에 국한된 부분(Partial) 보상 체계**임을 명확히 드러내는 핵심적 관측 결과이다.

---

## 요약(Summary)

본 3×3 디지털 트윈 시뮬레이션은 로봇 아크 용접 공정에서 발생하는 다중 물리 외란 하에서 단일 조작 변수(토치 속도 $v$) 기반 폐루프 제어기의 거동을 정량적으로 재현하였다. 결과를 종합하면 다음과 같은 결론을 도출할 수 있다.

1. **부분 제어의 성공:** 제어기가 명시적으로 감시·보정하는 지표인 유효 열입력 $Q_{eff}$는 초기 과도 구간(약 5초) 이후 목표값에 안정적으로 수렴하였으며, 그 파생 지표인 비드·너겟 형상(깊이, 폭) 역시 안전 범위 내로 함께 안정화되었다. 이는 열입력 지배 방정식에 근거한 속도 보상 제어가 설계 목적에 부합하게 작동함을 실증한다.
2. **미보정 외란의 잔존:** 그러나 표면온도, 접촉각(굴곡), 전류·전압과 같은 원인계(原因系) 변수는 어떠한 보정도 받지 않는 순수 확률적 랜덤워크로 남아있어, 공정 후반으로 갈수록 위험 범주까지 자유롭게 발산하였다.
3. **결함 확률·강도 예측의 불안정성:** 결함 확률 및 이에 종속된 인장 전단 강도 지표는 접촉각이라는 미보정 변수에 의해 지배되어, "0%로 수렴하는 안전 상태"라는 이상적 결과가 아니라 전 구간에 걸친 고빈도 진동을 나타내었다. 이는 제어 알고리즘의 결함이라기보다, **현재 구현된 가드레일이 열입력 단일 축에 한정된 부분 보상 체계**라는 시스템 설계상의 범위 한정에서 비롯된 필연적 결과이다.

**공학적 함의:** 완전한(Full) 결함 억제를 달성하기 위해서는 현재의 속도 단일 축 보상 구조를 확장하여, 접촉각(굴곡) 편차를 실시간으로 감지하고 이를 상쇄할 수 있는 추가 조작 변수(예: 토치 자세 능동 보정, 위빙 진폭 가변 제어)를 갖춘 **다변수(Multi-variable) 폐루프 제어 아키텍처**로의 확장이 요구된다. 본 시뮬레이션은 이러한 확장의 필요성을 정량적 근거와 함께 제시하는 예비 연구(Preliminary Study)로서의 의의를 지닌다.